# Thí nghiệm đối chứng cho mô hình đồ thị (chạy ngầm trên Kaggle)

Notebook này huấn luyện 17 lần chạy để kiểm lại kết luận "thêm đồ thị không hơn PhoBERT" (phiếu CD1.12i, GAP-040):

| Nhóm | Thí nghiệm | Seed | Trả lời câu hỏi |
|---|---|---|---|
| R1 | `asgcn_khong_gcn`: ASGCN bỏ hai lớp GCN, giữ cách gộp attention theo khía cạnh | 42, 1337, 2024, 7, 123 | ASGCN thua PhoBERT vì cách gộp hay vì GCN |
| R2 | `phobert`, `asgcn`, `asgcn_linked`, `senticgcn` thêm hai seed | 7, 123 | Hiệu số có thật hay do seed (5 seed thay vì 3) |
| R3 | `phobert_fp32`: PhoBERT tắt độ chính xác hỗn hợp | 42, 1337, 2024 | Gỡ khác biệt fp16 với fp32 |
| Kiểm | `phobert_kaggle`: PhoBERT seed 42 chạy lại trên Kaggle | 42 | Đổi nền tảng (Colab sang Kaggle) làm kết quả lệch bao nhiêu |

Hai GPU của **T4 x2** chạy song song hai hàng, khoảng **8 đến 9 giờ**. Notebook tự dừng nhận việc mới ở mốc 11 giờ để kịp
đóng gói trước giới hạn 12 giờ của một phiên; việc không kịp được ghi "bỏ qua: hết giờ" và chạy nốt được ở lần sau.

## Cách chạy (một lần bấm, đóng trình duyệt được)

1. **Input** (khung bên phải): gắn **đúng dataset đã dùng khi đo chi phí**, có `visfd_train.jsonl`, `visfd_dev.jsonl`,
   `visfd_test.jsonl`. Các file `*_best.pt` trong dataset đó không dùng tới, để nguyên cũng được.
2. **Settings**: *Accelerator* = **GPU T4 x2**; *Internet* = **On**.
3. Góc trên bên phải: **Save Version** → chọn **Save & Run All (Commit)** → **Save**.
4. Khoảng 10 phút sau, mở trang version (biểu tượng lịch sử version) → tab *Logs*: phải thấy hai dòng
   `[GPU 0] bat dau ...` và `[GPU 1] bat dau ...`. Thấy rồi thì đóng trình duyệt.

**Chạy nốt lần sau** (chỉ khi có việc "bỏ qua: hết giờ" hoặc "lỗi"): *Add Input* → *Your Work* → chọn **output của
version trước** của chính notebook này, rồi Save Version lại. Việc nào đã có kết quả sẽ được chép sang và bỏ qua.

## 1. Mốc thời gian và kiểm GPU

In [ ]:
# Moc cua CA phien: hang doi o muc 5 dung moc nay de khong vuot gioi han 12 gio cua Kaggle.
import subprocess, time, torch

BAT_DAU_PHIEN = time.time()

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu or "(khong co nvidia-smi)")
assert torch.cuda.is_available(), "KHONG CO GPU. Settings > Accelerator > GPU T4 x2, roi Save Version lai."
TEN_GPU = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
# T4: bon mo hinh dung PhoBERT o Bang 4.3 deu huan luyen tren Tesla T4.
assert all("T4" in t for t in TEN_GPU), f"GPU DANG LA {TEN_GPU}, KHONG PHAI T4. Settings > Accelerator > GPU T4 x2."
SO_GPU = len(TEN_GPU)
print(f"OK: {SO_GPU} x {TEN_GPU[0]}")
if SO_GPU < 2:
    print("CHU Y: chi co 1 GPU. Hang doi chay tuan tu va KHONG kip het trong mot phien; "
          "viec khong kip duoc ghi 'bo qua: het gio', chay not o lan sau (xem dau notebook).")

## 2. Lấy mã từ GitHub

In [ ]:
import subprocess
from pathlib import Path

# Ma nguon va checkpoint de o /tmp, KHONG o /kaggle/working: moi thu trong /kaggle/working thanh output cua
# version (gioi han 20 GB), ma moi lan chay sinh khoang 2 GB checkpoint.
# Ten thu muc khong dat la "nsmgat": trung ten goi Python, import se nham sang thu muc nay.
REPO = Path("/tmp/nsmgat-repo")
URL = "https://github.com/2611326-NguyenMinhTrong/nsmgat-vietnamese-absa.git"

if not REPO.exists():
    r = subprocess.run(["git", "clone", "--quiet", URL, str(REPO)], capture_output=True, text=True)
    assert r.returncode == 0, "KHONG KEO DUOC MA. Bat Settings > Internet = On.\n" + r.stderr[-500:]
COMMIT = subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%H %s"],
                        capture_output=True, text=True).stdout.strip()
print(COMMIT)
for f in ("configs/asgcn_khong_gcn.yaml", "configs/phobert_fp32.yaml"):
    assert (REPO / f).exists(), f"MA TREN GITHUB CHUA CO {f}: push commit moi len GitHub roi Save Version lai."

## 3. Cài mã nguồn

Không cài `requirements.txt`: file đó có `torch`, cài đè lên bản torch có CUDA sẵn của Kaggle là hỏng GPU.

In [ ]:
%pip install -q transformers pyyaml
%pip install -q -e /tmp/nsmgat-repo
!python -c "import nsmgat, transformers, torch; print('nsmgat:', nsmgat.__file__); print('transformers', transformers.__version__, '| torch', torch.__version__)"

## 4. Nối dữ liệu, tải PhoBERT một lần

Ba file dữ liệu được kiểm số dòng và mã băm SHA-256, để chắc đó đúng là dữ liệu đã cho các con số của báo cáo.
PhoBERT được tải trước một lần, để hai tiến trình huấn luyện không cùng tải một lúc.

In [ ]:
import hashlib, os, subprocess
from pathlib import Path

GOC_VAO = Path("/kaggle/input")
# So dong va SHA-256 cua ba file o may hoc vien (do ngay 16/10/2026)
DU_LIEU = {
    "visfd_train.jsonl": (23872, "674db57d5a4969cb1cb1f0ea4757e94c9762d57a1e7d81488d4ab65971074790"),
    "visfd_dev.jsonl": (3316, "208653f1f02c423b0627036b2575716c8b3d6a859b10a69b11227c3204487060"),
    "visfd_test.jsonl": (6722, "f6d714f298b02476bb47adeb52ce6daf198d60f73cf8f7964e2253ebfea57a5e"),
}

for ten, (so_dong, bam) in DU_LIEU.items():
    ds = sorted(p for p in GOC_VAO.rglob(ten) if p.is_file())
    assert len(ds) == 1, f"CAN DUNG MOT FILE {ten} TRONG INPUT, DANG CO {len(ds)}: {[str(p) for p in ds]}"
    p = ds[0]
    dem = sum(1 for _ in p.open(encoding="utf-8"))
    sha = hashlib.sha256(p.read_bytes()).hexdigest()
    assert dem == so_dong and sha == bam, f"{ten} KHAC FILE O MAY: {dem} dong (can {so_dong}), sha256 {sha[:16]} (can {bam[:16]})."
    dich = REPO / "data/processed" / ten
    dich.parent.mkdir(parents=True, exist_ok=True)
    if dich.is_symlink() or dich.exists():
        dich.unlink()
    os.symlink(p, dich)
    print(f"{ten:20} {dem:>7,} dong, khop file o may  <- {p}")

r = subprocess.run(["python", "-c", "from transformers import AutoModel, AutoTokenizer; "
                    "AutoTokenizer.from_pretrained('vinai/phobert-base-v2'); "
                    "AutoModel.from_pretrained('vinai/phobert-base-v2'); print('Da tai PhoBERT')"],
                   capture_output=True, text=True)
assert r.returncode == 0, "KHONG TAI DUOC PhoBERT:\n" + r.stderr[-800:]
print(r.stdout.strip())

## 5. Danh sách việc

Thứ tự là thứ tự ưu tiên: nếu hết giờ thì việc ở cuối bị bỏ trước. Phút ước tính lấy từ Bảng 4.6 (trung bình 3 seed
trên T4); `phobert_fp32` chưa đo, ước bằng hai lần `phobert`.

In [ ]:
import json, shutil
from pathlib import Path

RA = Path("/kaggle/working/ket_qua")           # ket qua nho: metrics, du doan, nhat ky -> dong goi thanh ket_qua.zip
GIU = Path("/kaggle/working/checkpoints_giu")  # best.pt duoc giu lai (lon, tai ve hay khong tuy hoc vien)
RA.mkdir(parents=True, exist_ok=True)

SEED_GOC = (42, 1337, 2024)
SEED_MOI = (7, 123)
# (ten thi nghiem = --exp-name, config, --model, seed, phut uoc tinh tren mot T4)
VIEC = (
    [("asgcn_khong_gcn", "configs/asgcn_khong_gcn.yaml", "asgcn", s, 60) for s in SEED_GOC]
    + [("asgcn", "configs/asgcn.yaml", "asgcn", s, 62) for s in SEED_MOI]
    + [("phobert", "configs/phobert.yaml", "phobert", s, 24) for s in SEED_MOI]
    + [("phobert_kaggle", "configs/phobert.yaml", "phobert", 42, 24)]
    + [("asgcn_khong_gcn", "configs/asgcn_khong_gcn.yaml", "asgcn", s, 60) for s in SEED_MOI]
    + [("phobert_fp32", "configs/phobert_fp32.yaml", "phobert", s, 48) for s in SEED_GOC]
    + [("senticgcn", "configs/senticgcn.yaml", "senticgcn", s, 76) for s in SEED_MOI]
    + [("asgcn_linked", "configs/asgcn_linked.yaml", "asgcn", s, 63) for s in SEED_MOI]
)
# Moi checkpoint bi xoa sau khi danh gia, tru best.pt seed 42 cua hai mo hinh doi chung (de phan tich sau nay).
GIU_BEST = {("asgcn_khong_gcn", 42), ("phobert_fp32", 42)}

# Chay not: neu output cua version truoc duoc gan lam input, viec nao da co metrics.json thi chep sang va bo qua.
da_co = set()
for m in sorted(GOC_VAO.rglob("ket_qua/*/seed*/metrics.json")):
    exp, seed = m.parent.parent.name, int(m.parent.name[4:])
    dich = RA / exp / f"seed{seed}"
    if not (dich / "metrics.json").exists():
        shutil.copytree(m.parent, dich, dirs_exist_ok=True)
    da_co.add((exp, seed))
CON_LAI = [v for v in VIEC if (v[0], v[3]) not in da_co]

tong = sum(v[4] for v in CON_LAI) / 60
print(f"{len(VIEC)} viec; {len(VIEC) - len(CON_LAI)} viec da co tu version truoc; con {len(CON_LAI)}")
print(f"Uoc tinh {tong:.1f} gio tren mot GPU, khoang {tong / SO_GPU:.1f} gio tren {SO_GPU} GPU")
for v in CON_LAI:
    print(f"  {v[0]:16} seed {v[3]:<5} ~{v[4]} phut")

## 6. Chạy hàng đợi

Mỗi GPU một luồng, lấy việc kế tiếp khi xong việc trước. Mỗi việc là một lệnh `python -m nsmgat.train` riêng, nhật ký
ghi vào `ket_qua/_nhat_ky/`. Một việc chỉ được bắt đầu khi thời gian còn lại tới mốc 11 giờ ít nhất bằng 1,3 lần ước
tính; việc chạy quá hạn bị dừng và ghi lỗi. Ô này in một dòng mỗi khi bắt đầu hoặc xong một việc, và mỗi 15 phút.

In [ ]:
import json, os, queue, shutil, subprocess, threading, time

GIOI_HAN_GIO = 11.0   # Kaggle cat phien o 12 gio; chua mot gio de dong goi
HE_SO_AN_TOAN = 1.3
HAN_CHOT = BAT_DAU_PHIEN + GIOI_HAN_GIO * 3600
NHAT_KY = RA / "_nhat_ky"
NHAT_KY.mkdir(exist_ok=True)
TRANG_THAI = RA / "trang_thai.json"

hang = queue.Queue()
for v in CON_LAI:
    hang.put(v)
khoa = threading.Lock()
bang = json.loads(TRANG_THAI.read_text(encoding="utf-8")) if TRANG_THAI.exists() else []


def ghi(muc):
    with khoa:
        bang.append(muc)
        TRANG_THAI.write_text(json.dumps(bang, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"[GPU {muc['gpu']}] {muc['exp']}/seed{muc['seed']}: {muc['trang_thai']}"
          + (f" sau {muc['phut']} phut" if "phut" in muc else ""), flush=True)


def lam(gpu):
    while True:
        try:
            exp, cfg, model, seed, uoc = hang.get_nowait()
        except queue.Empty:
            return
        muc = {"exp": exp, "seed": seed, "gpu": gpu, "uoc_tinh_phut": uoc}
        con = HAN_CHOT - time.time()
        if con < uoc * 60 * HE_SO_AN_TOAN:
            muc["trang_thai"] = f"bo qua: het gio (con {con / 60:.0f} phut)"
            ghi(muc)
            continue
        print(f"[GPU {gpu}] bat dau {exp}/seed{seed} (~{uoc} phut)", flush=True)
        lenh = ["python", "-m", "nsmgat.train", "--config", cfg, "--model", model,
                "--exp-name", exp, "--seed", str(seed)]
        moi_truong = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
        t0 = time.time()
        with open(NHAT_KY / f"{exp}_seed{seed}.txt", "w", encoding="utf-8") as f:
            try:
                rc = subprocess.run(lenh, cwd=REPO, env=moi_truong, stdout=f, stderr=subprocess.STDOUT,
                                    timeout=max(60, con - 300)).returncode
            except subprocess.TimeoutExpired:
                rc = "qua han"
        muc["phut"] = round((time.time() - t0) / 60, 1)

        ket = REPO / "results" / exp / f"seed{seed}"
        if rc == 0 and (ket / "metrics.json").exists():
            dich = RA / exp / f"seed{seed}"
            dich.mkdir(parents=True, exist_ok=True)
            for ten in ("metrics.json", "predictions.jsonl"):
                shutil.copy(ket / ten, dich / ten)
            shutil.copy(REPO / "logs" / exp / f"seed{seed}.log", dich / "train.log")
            m = json.loads((ket / "metrics.json").read_text(encoding="utf-8"))
            muc.update(trang_thai="xong", macro_f1=m["test"]["macro_f1"], accuracy=m["test"]["accuracy"],
                       n_params=m["n_params"], config_hash=m["config_hash"])
        else:
            muc["trang_thai"] = f"loi (ma thoat {rc}), xem _nhat_ky/{exp}_seed{seed}.txt"

        ck = REPO / "checkpoints" / exp / f"seed{seed}"
        if muc["trang_thai"] == "xong" and (exp, seed) in GIU_BEST and (ck / "best.pt").exists():
            (GIU / exp / f"seed{seed}").mkdir(parents=True, exist_ok=True)
            shutil.move(str(ck / "best.pt"), str(GIU / exp / f"seed{seed}" / "best.pt"))
        shutil.rmtree(ck, ignore_errors=True)
        ghi(muc)


luong = [threading.Thread(target=lam, args=(g,)) for g in range(SO_GPU)]
for l in luong:
    l.start()
moc = time.time()
while any(l.is_alive() for l in luong):
    time.sleep(30)
    if time.time() - moc >= 900:
        moc = time.time()
        print(f"... {(moc - BAT_DAU_PHIEN) / 3600:.1f} gio tu luc mo phien; {len(bang)}/{len(CON_LAI)} viec da xu ly",
              flush=True)
print(f"HET HANG DOI sau {(time.time() - BAT_DAU_PHIEN) / 3600:.1f} gio")

## 7. Tổng hợp, tự kiểm và đóng gói

Bảng dưới gộp seed mới với ba seed cũ có sẵn trong repo, chỉ để xem nhanh. Phép kiểm định đầy đủ làm ở máy sau khi
mang kết quả về.

In [ ]:
import json, platform, shutil, statistics, subprocess, torch, transformers
from pathlib import Path

bang = json.loads(TRANG_THAI.read_text(encoding="utf-8")) if TRANG_THAI.exists() else []
print(f"{'thi nghiem':16} {'seed':>5}  {'macro-F1':>8} {'acc':>7} {'phut':>6}  trang thai")
for m in bang:
    f1 = f"{m['macro_f1'] * 100:8.2f}" if "macro_f1" in m else " " * 8
    acc = f"{m['accuracy'] * 100:7.2f}" if "accuracy" in m else " " * 7
    print(f"{m['exp']:16} {m['seed']:>5}  {f1} {acc} {m.get('phut', ''):>6}  {m['trang_thai']}")

print("\nMacro-F1 gop ca seed cu (tu repo) va seed moi:")
for exp in ("phobert", "phobert_fp32", "asgcn_khong_gcn", "asgcn", "asgcn_linked", "senticgcn", "phobert_kaggle"):
    gia_tri = {}
    for goc in (REPO / "results", RA):
        for p in sorted((goc / exp).glob("seed*/metrics.json")):
            gia_tri[p.parent.name] = json.loads(p.read_text(encoding="utf-8"))["test"]["macro_f1"] * 100
    if gia_tri:
        v = list(gia_tri.values())
        sd = f" +/- {statistics.stdev(v):.2f}" if len(v) > 1 else ""
        print(f"  {exp:16} {len(v)} seed: {statistics.mean(v):.2f}{sd}   ({', '.join(f'{k[4:]}={x:.2f}' for k, x in gia_tri.items())})")

kiem = []
for m in bang:
    if m["exp"] == "asgcn_khong_gcn" and "n_params" in m:
        kiem.append(("so tham so asgcn_khong_gcn = phobert (135.000.579)", m["n_params"] == 135000579))
        break
p_moi = RA / "phobert_kaggle/seed42/predictions.jsonl"
p_cu = REPO / "results/phobert/seed42/predictions.jsonl"
if p_moi.exists() and p_cu.exists():
    a = [json.loads(d) for d in p_cu.open(encoding="utf-8")]
    b = [json.loads(d) for d in p_moi.open(encoding="utf-8")]
    assert [x["uid"] for x in a] == [x["uid"] for x in b]
    khac = sum(x["y_pred"] != y["y_pred"] for x, y in zip(a, b))
    print(f"\nPhoBERT seed 42, Colab so voi Kaggle: {khac} / {len(a)} du doan khac ({khac / len(a) * 100:.2f} %); "
          "ba seed PhoBERT khac nhau trung binh 4,49 %")
for ten, dat in kiem:
    print(("OK   " if dat else "SAI  ") + ten)

moi_truong = {"commit": COMMIT, "gpu": TEN_GPU, "torch": torch.__version__, "transformers": transformers.__version__,
              "python": platform.python_version(), "so_gio_phien": round((time.time() - BAT_DAU_PHIEN) / 3600, 2),
              "ghi_chu": "hai lan chay song song tren hai GPU nen thoi gian huan luyen KHONG so duoc voi Bang 4.6"}
(RA / "moi_truong.json").write_text(json.dumps(moi_truong, ensure_ascii=False, indent=1), encoding="utf-8")
shutil.make_archive("/kaggle/working/ket_qua", "zip", "/kaggle/working", "ket_qua")
print("\nDa dong goi /kaggle/working/ket_qua.zip")

## 8. Mang về máy

1. Mở trang version vừa chạy → tab *Output* → tải **`ket_qua.zip`**. Thư mục `checkpoints_giu/` (khoảng 1,1 GB, hai
   file best.pt) chỉ tải khi muốn giữ; không bắt buộc.
2. Đặt `ket_qua.zip` vào `data\cache\kaggle_doi_chung\` của repo (thư mục này không đẩy lên GitHub).
3. Báo Claude. Claude kiểm từng file rồi mới đưa vào `results/`.

Không cần tải notebook về: Kaggle giữ bản chạy kèm nhật ký trong trang version.